# Note 02: Threads, Blocks, and Grids

A CUDA kernel runs many copies of the same code. This note explains how CUDA organizes that work into threads, blocks, and grids, then uses global indexing to assign each thread its own position.

## 1. What Does `<<<1, 1>>>` Mean?

A CUDA kernel launch looks like:

```cpp
hello_gpu<<<1, 1>>>();
```

The values inside:

```text
<<< >>>
```

tell CUDA how much parallel work to launch.

For now, the important rule is:

```text
<<< number of blocks, threads per block >>>
```

So:

```cpp
hello_gpu<<<1, 1>>>();
```

means:

```text
1 block
└── 1 thread
```

That means the GPU runs our kernel using one GPU thread.

---

# 2. Why Do We Need GPU Threads?

Suppose we want to add two arrays:

```text
A = [1, 2, 3, 4]
B = [5, 6, 7, 8]

C = [6, 8, 10, 12]
```

On a CPU, we might write:

```cpp
for (int i = 0; i < 4; i++) {
    C[i] = A[i] + B[i];
}
```

Conceptually, one CPU thread performs:

```text
A[0] + B[0]
       ↓
A[1] + B[1]
       ↓
A[2] + B[2]
       ↓
A[3] + B[3]
```

But these four additions do not depend on each other.

So CUDA lets us split the work between threads:

```text
Thread 0 → C[0] = A[0] + B[0]

Thread 1 → C[1] = A[1] + B[1]

Thread 2 → C[2] = A[2] + B[2]

Thread 3 → C[3] = A[3] + B[3]
```

This is the basic idea of GPU programming:

> Divide a large computation into many small independent pieces and execute them using many GPU threads.

---

# 3. Launching Multiple Threads

Consider this kernel:

```cpp
__global__ void hello_gpu() {
    printf("Hello from GPU thread!\n");
}
```

If we launch:

```cpp
hello_gpu<<<1, 4>>>();
```

we are asking CUDA for:

```text
1 block
4 threads
```

So the same kernel function executes once for each thread.

Conceptually:

```text
Thread 0 → hello_gpu()
Thread 1 → hello_gpu()
Thread 2 → hello_gpu()
Thread 3 → hello_gpu()
```

The output will therefore contain four messages.

---

# 4. How Does Each Thread Know Which Thread It Is?

CUDA provides built-in variables for every thread.

The first one is:

```cpp
threadIdx.x
```

This tells a thread its position inside its block.

Example:

```cpp
__global__ void hello_gpu() {
    printf(
        "Hello from thread %d\n",
        threadIdx.x
    );
}
```

Launch:

```cpp
hello_gpu<<<1, 4>>>();
```

Now the four threads see:

```text
threadIdx.x = 0
threadIdx.x = 1
threadIdx.x = 2
threadIdx.x = 3
```

So conceptually:

```text
Block 0

┌──────────┬──────────┬──────────┬──────────┐
│ Thread 0 │ Thread 1 │ Thread 2 │ Thread 3 │
└──────────┴──────────┴──────────┴──────────┘
```

Each thread runs the same kernel code but sees a different:

```cpp
threadIdx.x
```

This is how different threads can work on different pieces of data.

---

# 5. Why Do We Need Blocks?

CUDA does not organize all GPU threads as one giant list.

Instead, threads are grouped into:

```text
Grid
 ↓
Blocks
 ↓
Threads
```

So:

```cpp
kernel<<<3, 4>>>();
```

means:

```text
3 blocks
4 threads in each block
```

Total threads:

```text
3 × 4 = 12
```

Conceptually:

```text
GRID

Block 0
┌────┬────┬────┬────┐
│ T0 │ T1 │ T2 │ T3 │
└────┴────┴────┴────┘

Block 1
┌────┬────┬────┬────┐
│ T0 │ T1 │ T2 │ T3 │
└────┴────┴────┴────┘

Block 2
┌────┬────┬────┬────┐
│ T0 │ T1 │ T2 │ T3 │
└────┴────┴────┴────┘
```

Notice something important:

Every block has its own:

```text
Thread 0
Thread 1
Thread 2
Thread 3
```

So `threadIdx.x` is only unique **inside one block**.

---

# 6. How Does a Thread Know Which Block It Belongs To?

CUDA provides:

```cpp
blockIdx.x
```

This tells a thread which block it belongs to.

For:

```cpp
kernel<<<3, 4>>>();
```

the block IDs are:

```text
Block 0
Block 1
Block 2
```

And inside each block:

```text
threadIdx.x = 0, 1, 2, 3
```

So:

```text
Block 0:
T0 T1 T2 T3

Block 1:
T0 T1 T2 T3

Block 2:
T0 T1 T2 T3
```

We can inspect both:

```cpp
__global__ void hello_gpu() {

    printf(
        "Block %d, Thread %d\n",
        blockIdx.x,
        threadIdx.x
    );
}
```

Launch:

```cpp
hello_gpu<<<3, 4>>>();
```

This creates 12 GPU threads.

---

# 7. Four CUDA Variables You Will See Constantly

These are extremely important:

```cpp
threadIdx.x
blockIdx.x
blockDim.x
gridDim.x
```

For:

```cpp
kernel<<<3, 4>>>();
```

we get:

```text
gridDim.x  = 3
blockDim.x = 4
```

And each thread has its own:

```text
blockIdx.x
threadIdx.x
```

Meaning:

```text
blockIdx.x
    → Which block am I inside?

threadIdx.x
    → Which thread am I inside that block?

blockDim.x
    → How many threads are inside each block?

gridDim.x
    → How many blocks exist in the grid?
```

---

# 8. The Global Thread ID

Suppose:

```cpp
kernel<<<3, 4>>>();
```

We have 12 threads in total.

We often want every thread to calculate a unique ID:

```text
0
1
2
3
4
5
...
11
```

The standard CUDA formula is:

```cpp
int i =
    blockIdx.x * blockDim.x
    + threadIdx.x;
```

Do not blindly memorize this.

Understand where it comes from.

---

## Example: Block 0, Thread 2

We know:

```text
blockIdx.x = 0
blockDim.x = 4
threadIdx.x = 2
```

So:

```text
i = 0 * 4 + 2
i = 2
```

Global thread ID:

```text
2
```

---

## Example: Block 1, Thread 2

```text
blockIdx.x = 1
blockDim.x = 4
threadIdx.x = 2
```

So:

```text
i = 1 * 4 + 2
i = 6
```

Global thread ID:

```text
6
```

---

## Example: Block 2, Thread 3

```text
blockIdx.x = 2
blockDim.x = 4
threadIdx.x = 3
```

So:

```text
i = 2 * 4 + 3
i = 11
```

---

# 9. Visualizing the Global IDs

For:

```cpp
kernel<<<3, 4>>>();
```

we get:

```text
Block 0
T0  T1  T2  T3
 ↓   ↓   ↓   ↓
 0   1   2   3


Block 1
T0  T1  T2  T3
 ↓   ↓   ↓   ↓
 4   5   6   7


Block 2
T0  T1  T2  T3
 ↓   ↓   ↓   ↓
 8   9  10  11
```

That is exactly what this formula creates:

```cpp
int i =
    blockIdx.x * blockDim.x
    + threadIdx.x;
```

The formula answers:

> Which piece of the overall problem belongs to this thread?

This pattern appears constantly in CUDA kernels.

---

# 10. Colab Experiment

Create:

```cpp
%%writefile threads.cu

#include <iostream>
#include <cuda_runtime.h>

__global__ void show_threads() {

    int i =
        blockIdx.x * blockDim.x
        + threadIdx.x;

    printf(
        "Block: %d | Thread: %d | Global ID: %d\n",
        blockIdx.x,
        threadIdx.x,
        i
    );
}

int main() {

    show_threads<<<3, 4>>>();

    cudaDeviceSynchronize();

    return 0;
}
```

Compile:

```bash
!nvcc threads.cu -o threads
```

Run:

```bash
!./threads
```

You should see global IDs covering:

```text
0 through 11
```

The lines may not always print in numerical order.

That is normal.

GPU threads execute concurrently, so we should not depend on print ordering.

---

# 11. Current Mental Model

CUDA execution hierarchy:

```text
GRID
 │
 ├── BLOCK
 │     ├── THREAD
 │     ├── THREAD
 │     └── THREAD
 │
 ├── BLOCK
 │     ├── THREAD
 │     ├── THREAD
 │     └── THREAD
 │
 └── ...
```

A kernel launch:

```cpp
kernel<<<blocks, threads_per_block>>>();
```

creates:

```text
total threads
=
number of blocks
×
threads per block
```

Every thread runs the same kernel code.

But each thread can determine its identity using:

```cpp
blockIdx.x
threadIdx.x
```

and calculate a globally unique position with:

```cpp
int i =
    blockIdx.x * blockDim.x
    + threadIdx.x;
```

---

# 12. What We Learned

We now understand:

* what `<<<1, 1>>>` means
* what a CUDA thread is
* why many threads can run the same kernel
* what a block is
* what a grid is
* `threadIdx.x`
* `blockIdx.x`
* `blockDim.x`
* `gridDim.x`
* how to calculate a global thread ID
* why global indexing is necessary for parallel data processing
* why GPU print output order should not be relied upon

---



In [ ]:
%%writefile threads_blocks.cu

#include <iostream>
#include <cuda_runtime.h>

__global__ void hello_gpu() {

    printf(
        "Block %d, Thread %d\n",
        blockIdx.x,
        threadIdx.x
    );
}

int main(){
  hello_gpu<<<3,4>>>();
  cudaDeviceSynchronize();
  return 0;
}

Writing threads_blocks.cu


In [ ]:
!nvcc threads_blocks.cu -o threads_blocks
!./threads_blocks

Block 1, Thread 0
Block 1, Thread 1
Block 1, Thread 2
Block 1, Thread 3
Block 0, Thread 0
Block 0, Thread 1
Block 0, Thread 2
Block 0, Thread 3
Block 2, Thread 0
Block 2, Thread 1
Block 2, Thread 2
Block 2, Thread 3


In [ ]:
%%writefile threads.cu

#include <iostream>
#include <cuda_runtime.h>

__global__  void threads(){
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  printf(
        "Block: %d | Thread: %d | Global ID: %d\n",
        blockIdx.x,
        threadIdx.x,
        i
    );
}

int main(){
threads<<<3,4>>>();
cudaDeviceSynchronize();
}

Writing threads.cu


In [ ]:
!nvcc threads.cu -o threads #there is no order of parallel computation thats why the final global ids are randomly arranged
!./threads

Block: 1 | Thread: 0 | Global ID: 4
Block: 1 | Thread: 1 | Global ID: 5
Block: 1 | Thread: 2 | Global ID: 6
Block: 1 | Thread: 3 | Global ID: 7
Block: 0 | Thread: 0 | Global ID: 0
Block: 0 | Thread: 1 | Global ID: 1
Block: 0 | Thread: 2 | Global ID: 2
Block: 0 | Thread: 3 | Global ID: 3
Block: 2 | Thread: 0 | Global ID: 8
Block: 2 | Thread: 1 | Global ID: 9
Block: 2 | Thread: 2 | Global ID: 10
Block: 2 | Thread: 3 | Global ID: 11
